# Concept-Monosemanticity: Top-Patches & Tversky-MS pro Concept

Fragestellung: feuert ein einzelnes SAE-Concept (eine Latent-Dimension im Dictionary)
konsistent fuer eine Semantik (z.B. "roter Fluegel"), oder feuert es fuer mehrere
unterschiedliche Semantiken (Polysemanticity)?

Ansatz:
1. Fertig trainierten Checkpoint laden, SAE ueber den gesamten Test-Split laufen lassen
   und die **per-Patch** Concept-Aktivierungen sammeln (nicht ueber das Bild gepoolt).
2. Fuer jedes Concept die `TOP_K` am staerksten aktivierenden Patches ueber den ganzen
   Split finden und die zugehoerigen Bildausschnitte anzeigen -- rein visuelle Pruefung,
   ob diese Ausschnitte inhaltlich zusammenpassen.
3. Zusaetzlich quantitativ: den bereits vorhandenen Tversky-MS-Score
   (`src/cbm_msae_lab/metrics/tversky_ms.py`) **pro Latent** (nicht nur aggregiert)
   berechnen, um die polysemantischsten Concepts automatisch zu finden, statt alle
   8192 einzeln durchzuklicken.

Arbeitet immer mit einem einzelnen Checkpoint (`CHECKPOINT_PATH` unten); fuer einen
Vergleich zwischen Checkpoints das Notebook mit anderem Pfad erneut laufen lassen.

In [1]:
import os
from pathlib import Path

import hydra.utils
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from cbm_msae_lab.checkpointing import load_for_reproduction
from cbm_msae_lab.config_schema import register_configs
from cbm_msae_lab.metrics.tversky_ms import _pack_bitset, _tversky_ms_per_latent

# conf/*.yaml-Pfade (checkpoints/, cache/) sind relativ zum Repo-Root, nicht zu
# notebooks/ -- gleiche Annahme wie in scripts/*.py und attention_maps.ipynb.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

register_configs()
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

## Parameter

`CHECKPOINT_PATH` fuer einen anderen Checkpoint/Vergleich anpassen.

In [4]:
CHECKPOINT_PATH = "outputs/checkpoints/cub_cfm_vanilla/best.pt"  # <- anpassen
SPLIT = "test"
TOP_K = 16  # Top-Patches pro Concept (4x4-Grid)
BATCH_SIZE = 64
NUM_WORKERS = 4
CROP_MARGIN = 8  # Pixel, um den reinen Patch-Crop herum (ein 16x16-px-Patch ist sonst kaum erkennbar)

# Concepts, die im Detail angeschaut werden sollen. Leer lassen, um automatisch die laut
# Tversky-MS-Score polysemantischsten Concepts zu waehlen (siehe Ranking-Zelle unten).
CONCEPTS_TO_INSPECT: list[int] = []
N_AUTO_CONCEPTS = 6  # nur relevant, wenn CONCEPTS_TO_INSPECT leer ist

# Tversky-MS-Parameter, siehe metrics/tversky_ms.py
TMS_MAX_PAIRS = 1000
TMS_ALPHA = 1.0
TMS_BETA = 1.0
TMS_MIN_ACTIVE = 20  # Latents mit kleinerem aktivem Set werden im Ranking ignoriert (zu verrauscht)

## Checkpoint + Dataset laden

`load_for_reproduction` baut Encoder/Upsampler/SAE exakt aus der im Checkpoint gespeicherten
Config wieder auf -- `cfg.dataset` passt also automatisch zum trainierten Modell.

In [5]:
cfg, pipeline = load_for_reproduction(CHECKPOINT_PATH, device=device)
pipeline.eval()

dataset = hydra.utils.instantiate(cfg.dataset, split=SPLIT)
# shuffle=False ist Pflicht: der Dataset-Index muss sich spaeter auf dataset.paths[idx]
# zurueckfuehren lassen (gleiche Konvention wie scripts/extract_raw_features.py).
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

image_size = dataset[0][0].shape[-1]
dict_size = cfg.sae.dict_size
print(f"{len(dataset)} Bilder, image_size={image_size}, dict_size={dict_size}")

5794 Bilder, image_size=224, dict_size=8192


## Forward-Pass ueber den ganzen Split

Fuer jedes Concept wird ein Streaming-Top-K ueber `(image_idx, row, col, value)` gepflegt --
bei `dict_size` im vierstelligen Bereich und ~1M Patches im Test-Split waere das Speichern
aller Patch-Aktivierungen zu teuer, daher wird pro Batch gemerged.

Zusaetzlich werden die **max-gepoolten** Aktivierungen pro Bild gesammelt (wie in
`scripts/evaluate_task_accuracy.py::extract_pooled_representations`), als Grundlage fuer
den Tversky-MS-Score.

In [7]:
# Laufender Top-K-Buffer pro Concept: (values[dict_size, TOP_K], image_idx[..], row[..], col[..])
top_values = torch.full((dict_size, TOP_K), -float("inf"))
top_image_idx = torch.full((dict_size, TOP_K), -1, dtype=torch.long)
top_row = torch.full((dict_size, TOP_K), -1, dtype=torch.long)
top_col = torch.full((dict_size, TOP_K), -1, dtype=torch.long)

pooled_chunks: list[torch.Tensor] = []  # je [B, dict_size], max-gepoolt pro Bild


@torch.no_grad()
def update_top_k(concept_values: torch.Tensor, image_idx: torch.Tensor, row: torch.Tensor, col: torch.Tensor) -> None:
    """concept_values/image_idx/row/col: [dict_size, B*H*W] (gleiche Reihenfolge). Merged die
    Kandidaten dieses Batches in den globalen Top-K-Buffer pro Concept."""
    cand_values = torch.cat([top_values, concept_values], dim=1)
    cand_image_idx = torch.cat([top_image_idx, image_idx], dim=1)
    cand_row = torch.cat([top_row, row], dim=1)
    cand_col = torch.cat([top_col, col], dim=1)

    keep = cand_values.topk(TOP_K, dim=1).indices  # [dict_size, TOP_K]
    top_values.copy_(cand_values.gather(1, keep))
    top_image_idx.copy_(cand_image_idx.gather(1, keep))
    top_row.copy_(cand_row.gather(1, keep))
    top_col.copy_(cand_col.gather(1, keep))

from tqdm import tqdm

global_offset = 0
with torch.no_grad():
    for images, _labels, _idx in tqdm(loader):
        images = images.to(device)
        features = pipeline.project(images)  # [B, C_sae, H, W]
        B, C, H, W = features.shape
        x_flat = features.permute(0, 2, 3, 1).reshape(B * H * W, C)

        z_flat, _active, _a_flat = pipeline.sae.encode(x_flat, return_active=True, use_threshold=True)
        # z_flat: [B*H*W, dict_size] -- thresholded, per-Patch

        z_img = z_flat.reshape(B, H * W, dict_size)  # [B, P, dict_size]
        pooled_chunks.append(z_img.max(dim=1).values.cpu())  # [B, dict_size]

        rows_grid = torch.arange(H, device=device).repeat_interleave(W)  # [P]
        cols_grid = torch.arange(W, device=device).repeat(H)  # [P]
        image_idx_grid = global_offset + torch.arange(B, device=device)  # [B]

        concept_values = z_img.permute(2, 0, 1).reshape(dict_size, B * H * W).cpu()
        image_idx_bp = image_idx_grid.view(B, 1).expand(B, H * W).reshape(1, B * H * W).expand(dict_size, -1).cpu()
        row_bp = rows_grid.view(1, H * W).expand(B, H * W).reshape(1, B * H * W).expand(dict_size, -1).cpu()
        col_bp = cols_grid.view(1, H * W).expand(B, H * W).reshape(1, B * H * W).expand(dict_size, -1).cpu()

        update_top_k(concept_values, image_idx_bp, row_bp, col_bp)
        global_offset += B

grid_h, grid_w = H, W
pooled = torch.cat(pooled_chunks, dim=0)  # [N, dict_size]
print(f"grid={grid_h}x{grid_w}, gepoolte Aktivierungen: {pooled.shape}")

  0%|          | 0/91 [00:40<?, ?it/s]


KeyboardInterrupt: 

## Tversky-MS-Score pro Concept

Niedriger Score bei ausreichend grossem aktivem Set = das Concept feuert bei Bildern mit
sehr unterschiedlichen sonstigen Ko-Aktivierungsmustern -- ein Hinweis auf Polysemanticity.

In [ ]:
acts_np = pooled.numpy()
binary = acts_np > acts_np.mean(axis=0, keepdims=True)
packed = _pack_bitset(binary)

tms_scores, active_set_sizes = _tversky_ms_per_latent(
    packed, dict_size, TMS_MAX_PAIRS, TMS_ALPHA, TMS_BETA, seed=0, min_active=TMS_MIN_ACTIVE
)

valid = active_set_sizes >= TMS_MIN_ACTIVE
print(f"{valid.sum()} / {dict_size} Concepts mit aktivem Set >= {TMS_MIN_ACTIVE}")

plt.figure(figsize=(6, 3))
plt.hist(tms_scores[valid], bins=50)
plt.xlabel("Tversky-MS-Score")
plt.ylabel("# Concepts")
plt.title("Verteilung der Per-Latent-TMS-Scores")
plt.tight_layout()

In [ ]:
if CONCEPTS_TO_INSPECT:
    concepts = CONCEPTS_TO_INSPECT
else:
    # niedrigster TMS-Score zuerst = polysemantischste Concepts, nur unter validen Latents
    order = np.argsort(np.where(valid, tms_scores, np.inf))
    concepts = order[:N_AUTO_CONCEPTS].tolist()

for k in concepts:
    print(f"concept {k:5d}  tms={tms_scores[k]:.3f}  active_set={active_set_sizes[k]}")

## Top-Patches pro Concept visualisieren

Patch `(row, col)` -> Pixel-Box: `stride = image_size // grid_w`, Crop um `CROP_MARGIN`
Pixel erweitert (ein reiner 16x16-px-Patch ist sonst kaum zu erkennen).

In [ ]:
def crop_patch(image: torch.Tensor, row: int, col: int, grid_h: int, grid_w: int, margin: int) -> np.ndarray:
    """image: [3, image_size, image_size] in [0,1] -> HWC numpy crop around patch (row, col)."""
    image_size = image.shape[-1]
    stride_h = image_size // grid_h
    stride_w = image_size // grid_w
    y0 = max(0, row * stride_h - margin)
    y1 = min(image_size, (row + 1) * stride_h + margin)
    x0 = max(0, col * stride_w - margin)
    x1 = min(image_size, (col + 1) * stride_w + margin)
    return image[:, y0:y1, x0:x1].permute(1, 2, 0).numpy()


def show_top_patches(concept: int, n_cols: int = 4) -> None:
    n_rows = (TOP_K + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2 * n_cols, 2.2 * n_rows))
    fig.suptitle(f"Concept {concept} (tms={tms_scores[concept]:.3f}, active_set={active_set_sizes[concept]})")
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i >= TOP_K:
            continue
        img_idx = int(top_image_idx[concept, i])
        row = int(top_row[concept, i])
        col = int(top_col[concept, i])
        value = float(top_values[concept, i])
        image, _label, _idx = dataset[img_idx]
        crop = crop_patch(image, row, col, grid_h, grid_w, CROP_MARGIN)
        ax.imshow(crop)
        ax.set_title(f"idx={img_idx}\nact={value:.2f}", fontsize=8)
    fig.tight_layout()


for k in concepts:
    show_top_patches(k)

## Naechste Schritte

- `CHECKPOINT_PATH` auf einen anderen fertig trainierten Checkpoint setzen, um Modelle zu
  vergleichen (z.B. unterschiedliche `k`/`dict_size`/Loss-Konfigurationen).
- `CONCEPTS_TO_INSPECT` gezielt auf Concepts setzen, die im Training ungewoehnlich aussahen
  (z.B. sehr hohe Aktivierungsfrequenz).
- `TMS_ALPHA`/`TMS_BETA` variieren (Tversky-Asymmetrie) oder `TMS_MIN_ACTIVE` senken/erhoehen,
  um zu sehen, wie stabil das Ranking ist.
- Falls die Top-Patches eines Concepts fast alle aus demselben Bild stammen, ist das eher ein
  Hinweis auf einen "dead"/kaum genutzten Bereich des Dictionaries als auf Polysemanticity --
  `active_set_sizes[k]` daneben pruefen.